# Training Lora From Corpus Version 2 (Big)
This adds stop tokens to the training

# Load model with Unsloth patching

In [ ]:
from unsloth import FastLanguageModel

# different model options here
# Qwen/Qwen3-32B
# Qwen/Qwen3-32B-Instruct
# deepseek-ai/DeepSeek-R1-Distill-Qwen-32

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen3-30B-A3B",
    max_seq_length=1024,
    load_in_4bit=True,
)

print("Loaded model in 4-bit ✅")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
INFO 10-10 14:07:12 [__init__.py:216] Automatically detected platform cuda.
🦥 Unsloth Zoo will now patch everything to make training faster!


Unsloth: Failed to create directory `unsloth_compiled_cache` because [Errno 13] Permission denied: 'unsloth_compiled_cache'


==((====))==  Unsloth 2025.9.11: Fast Qwen3_Moe patching. Transformers: 4.56.2. vLLM: 0.10.2.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.318 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33+5146f2a.d20251002. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


# Apply LoRa adapter

In [2]:
peft_model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    lora_alpha=64,
    lora_dropout=0.07,
    target_modules=["q_proj","v_proj","o_proj","k_proj"],
    bias="none",
    use_gradient_checkpointing=True,
)

tokenizer.bos_token = None
tokenizer.eos_token = "</s>"  # Qwen3’s typical EOS token
tokenizer.pad_token = tokenizer.eos_token  # Common practice
model.config.bos_token_id = tokenizer.bos_token_id
model.config.eos_token_id = tokenizer.eos_token_id
model.config.pad_token_id = tokenizer.pad_token_id
peft_model.config.bos_token_id = tokenizer.bos_token_id
peft_model.config.eos_token_id = tokenizer.eos_token_id
peft_model.config.pad_token_id = tokenizer.pad_token_id

print("Loaded peft model ✅")


Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.07.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.


Unsloth: Making `model.base_model.model.model` require gradients
Loaded peft model ✅


# Load the dataset from corpus

In [3]:
import os 

#CORPUS_DIR = "/storage/corpus/wtk_archive_with_stops"
CORPUS_DIR = "/storage/corpus/ai_corpus_slimmer_clean"

BLOCK_SIZE = 1024  # max tokens per chunk

tok = tokenizer 

# Ensure EOS/PAD exist and are consistent
added = False
if tok.eos_token is None:
    tok.add_special_tokens({"eos_token": "</s>"})
    added = True
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
    added = True
if added:
    model.resize_token_embeddings(len(tok))

# -----------------------
# 2) Load raw text files (no EOS strings here)
# -----------------------
def load_txt_corpus(directory):
    texts = []
    for filename in os.listdir(directory):
        if not filename.endswith(".txt"):
            continue
        path = os.path.join(directory, filename)
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            txt = f.read().strip()
            if txt:
                texts.append(txt)
    return texts

raw_texts = load_txt_corpus(CORPUS_DIR)
print(f"Loaded {len(raw_texts)} files from {CORPUS_DIR} ✅")

Loaded 11882 files from /storage/corpus/ai_corpus_slimmer_clean ✅


# Tokenize with EOS appended (token id, not string)
We’ll build one long stream of ids and then pack into BLOCK_SIZE chunks.

In [4]:
from datasets import Dataset

def tokenize_append_eos(texts):
    # batch tokenize; append eos token string so tokenizer emits eos_token_id
    # Alternatively: add eos id manually after each example (equivalent).
    enc = tok(texts, add_special_tokens=False)
    input_ids = []
    for ids in enc["input_ids"]:
        input_ids.extend(ids)
        if tok.eos_token_id is not None:
            input_ids.append(tok.eos_token_id)
    return input_ids

flat_ids = tokenize_append_eos(raw_texts)

# -----------------------
# 4) Pack into fixed-length blocks (no cross-doc bleed because we injected EOS)
# -----------------------
def pack_ids_to_blocks(ids, block_size):
    blocks = []
    for i in range(0, len(ids) - block_size + 1, block_size):
        chunk = ids[i : i + block_size]
        blocks.append({"input_ids": chunk, "attention_mask": [1] * len(chunk)})
    return Dataset.from_list(blocks)

train_dataset = pack_ids_to_blocks(flat_ids, BLOCK_SIZE)
print(f"Prepared {len(train_dataset)} packed training chunks of {BLOCK_SIZE} tokens ✅")


Prepared 31156 packed training chunks of 1024 tokens ✅


# Init Trainer Params

In [7]:
import os
from transformers import TrainingArguments
from trl import SFTTrainer

# 1) Absolute, writable, persistent output dir
OUTPUT_DIR = "/workspace/wtk-qwen3-beta-slim-lora-v3"

# 2) Build explicit TrainingArguments (NO dict here)
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    overwrite_output_dir=True,      # reuse dir safely
    resume_from_checkpoint=True,    # set False if you want a totally clean start
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    lr_scheduler_type="cosine",
    learning_rate=1e-4,
    warmup_ratio=0.03, 
    fp16=False,
    bf16=True,
    logging_steps=10,
    report_to="none",
    remove_unused_columns=False,
    metric_for_best_model="loss",
    greater_is_better=False,
    max_grad_norm=1.0,
    dataloader_num_workers=0,       # more stable on mounted storage
)

# 3) Build the trainer
trainer = SFTTrainer(
    model=peft_model,
    tokenizer=tok,
    train_dataset=train_dataset,
    max_seq_length=BLOCK_SIZE,
    args=training_args,
)

print(f"Created SFTTrainer ✅")


Created SFTTrainer ✅


# Train using SFTTrainer (new)

In [5]:
# 4) Start training (this might take a while)
trainer.train()
print("Training complete ✅")

# 5) Save trained model to storage
trainer.model.save_pretrained(OUTPUT_DIR)
tok.save_pretrained(OUTPUT_DIR)

print("Training results saved ✅")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 31,156 | Num Epochs = 2 | Total steps = 3,896
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 576,454,656 of 31,108,577,280 (1.85% trained)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got fo

Unsloth: Will smartly offload gradients to save VRAM!


Exception ignored in: <function ExactWeakKeyDictionary.__setitem__.<locals>.<lambda> at 0x7f37db7d6fc0>
Traceback (most recent call last):
  File "/opt/conda/lib/python3.11/site-packages/torch/_dynamo/utils.py", line 954, in <lambda>
    self.refs[idx] = weakref.ref(key, lambda ref: self._remove_id(idx))

KeyboardInterrupt: 


KeyboardInterrupt: 

# Resume training using SFTTrainer (only use if resuming)

In [8]:
CHECKPOINT_DIR = OUTPUT_DIR + "/" + "checkpoint-500"

#4) Start training (this might take a while)
print(f"Resuming from checkpoint: {CHECKPOINT_DIR}")
trainer.train(resume_from_checkpoint=CHECKPOINT_DIR)
print("Training complete ✅")

# 5) Save trained model to storage
trainer.model.save_pretrained(OUTPUT_DIR)
tok.save_pretrained(OUTPUT_DIR)

print("Training results saved ✅")

Resuming from checkpoint: /workspace/wtk-qwen3-beta-slim-lora-v3/checkpoint-500


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 31,156 | Num Epochs = 1 | Total steps = 1,948
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 26,738,688 of 30,558,861,312 (0.09% trained)


InternalTorchDynamoError: ValueError: stoi


# Push LoRa to Huggingface

In [1]:
from huggingface_hub import HfApi, upload_folder

repo_id = "peers-ai/deepseek-7b-my-lora1-with-stops"
folder = "lora-txt-training2"  # contains adapter_config.json & adapter_model.bin

api = HfApi()
# create the repo if it doesn't exist
api.create_repo(repo_id, repo_type="model", private=True, exist_ok=True)

# upload all files in the folder
upload_folder(
    repo_id=repo_id,
    folder_path=folder,
    repo_type="model",
)
print(f"✅ Uploaded to https://huggingface.co/{repo_id}")


Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

  ...training2/adapter_model.safetensors:  97%|#########7| 15.3MB / 15.7MB            

✅ Uploaded to https://huggingface.co/peers-ai/deepseek-7b-my-lora1-with-stops
